# 01 · GenAI Basics — your first calls to the reasoner

**Agentic AI for Actuaries** · IFoA Workshop · 10 July 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 1, Part 2 (The Reasoner). 
**You will:** make your first Gemini API call, practise the CCCE prompt discipline, watch a hallucination happen on demand, and get guaranteed-parseable JSON out of an LLM.

**Setup (2 minutes):**
1. Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
2. In Colab, click the **key icon** (left sidebar) → *Add new secret* → name it `GOOGLE_API_KEY`, paste the key, toggle notebook access ON.
3. Run the cells top to bottom (`Runtime → Run all` after setup).

In [ ]:
%pip install -q -U google-genai

## §1 · Auth — the key never appears in the notebook
Colab Secrets keeps the key out of the notebook file. This is the same hygiene you will use for every agent you ship: secrets live in a store, never in code.

In [ ]:
import os
from google import genai
from google.colab import userdata   # Colab-only; see comment below for local Jupyter

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# Local Jupyter alternative:
#   os.environ["GOOGLE_API_KEY"] = "..."  # or use python-dotenv

client = genai.Client()
MODEL = "gemini-3.1-flash-lite"   # PINNED — silent model drift is an audit failure
print("Client ready, model pinned to:", MODEL)

## §2 · First call — define IBNR for a board member

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a non-actuarial board member, in one line.",
)
print(response.text)
print("\n--- usage ---")
print(response.usage_metadata)   # token counts: you will care about these when agents multiply call volume

## §3 · CCCE — Clarity, Context, Constraints, Examples
The prompt below is the worked example from the slides: an IBNR commentary for ABC Health Q3 2024. Each bracketed fragment does exactly one job — edit any part without breaking the others.

**Exercise:** delete the Constraints block, re-run, and compare. Then rewrite the prompt for *your* line of business.

In [ ]:
ccce_prompt = """
[Clarity] Write a two-paragraph commentary on the IBNR result for ABC Health Q3 2024.
[Context] Indemnity health book. Chain-ladder ultimate INR 186 Cr vs prior estimate INR 172 Cr.
Q3 saw a hospital network strike in two states.
[Constraints] Audience: appointed actuary peer-review meeting. Max 180 words.
Do not invent figures. Cite only the figures provided above.
[Example] Voice to match: "The Q2 ultimate of INR 164 Cr increased to INR 172 Cr after the network
expansion in Tier 2 cities..."
"""
resp = client.models.generate_content(model=MODEL, contents=ccce_prompt)
print(resp.text)

### §3.1 · Demo 1 — the vague version, for contrast
Run the deliberately vague prompt below, then re-run the CCCE version above and **diff the outputs**. Same model, same cost — the entire quality delta is the prompt.

In [ ]:
vague = "Write about IBNR for our board."
print(client.models.generate_content(model=MODEL, contents=vague).text[:800])
# Expect: a generic essay that INVENTS plausible numbers (we gave it none)
# and lands in a register somewhere between textbook and LinkedIn.

### §3.2 · Demo 2 — one fact, two audiences
Audience is a prompt parameter. Same reserve-strengthening fact, rendered for a board member and for a new student. Note: the model *dresses* the fact we supply — it does not source it.

In [ ]:
fact = ("We strengthened motor BI reserves by INR 42 Cr "
        "following the new tribunal award benchmarks.")

for audience, style in [
    ("board member", "2 sentences, business impact first, no jargon"),
    ("new actuarial student", "4 sentences, explain WHY tribunal awards drive BI reserves, define terms"),
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Explain: {fact} For a {audience}. {style}")
    print(f"--- {audience.upper()} ---\n{r.text}\n")

### §3.3 · Demo 3 — few-shot examples tame formatting
Show, don't tell: two worked examples buy you the delimiter, the casing, the arrow convention, and no chatty preamble. **Exercise:** feed it a genuinely weird input and see whether the pattern holds.

In [ ]:
prompt = """Convert each change to the format of the examples.

EXAMPLES
In: We moved lapse from 6% to 5.5% for durations 2+.
Out: LAPSE | dur 2+ | 6.0% -> 5.5%
In: Expense inflation up 50bps.
Out: EXPENSE_INFL | all | +50bps

NOW CONVERT
In: Mortality improvement for males 45-60 moves from 1.5% to 1.25%.
Out:"""
print(client.models.generate_content(model=MODEL, contents=prompt).text)

### §3.4 · Demo 4 — step-by-step reasoning (with a warning label)
Asking for steps improves reliability — it does **not** guarantee it. Re-run this cell three times: do the running totals stay identical? This is why the afternoon's agent does arithmetic in *Python* and lets Gemini narrate.

In [ ]:
prompt = """A motor policy has base premium INR 6,500 with relativities:
vehicle age 6-9yrs = 1.15, SUV = 1.20, Tier2 = 1.00, NCB 35% = 0.65.
Walk through the premium calculation STEP BY STEP, showing the running
total after each factor, then state the final premium."""
print(client.models.generate_content(model=MODEL, contents=prompt).text)
# Check by hand: 6500 * 1.15 * 1.20 * 1.00 * 0.65 = 5,830.50

### §3.5 · Demo review — the habit that IS the skill
1. CCCE moved quality more than a model upgrade would — specification beats horsepower.
2. Register control is leverage, but the model dresses facts; it doesn't source them.
3. Few-shot is a formatting contract — stress-test it before relying on it.
4. Step-by-step is transparency, not verified arithmetic.

**The loop:** prompt → output → review → edit prompt — the same loop you'll run on agent traces this afternoon.

## §4 · The hallucination demo — run it, believe it
We ask for a regulation that **does not exist**. The model will not say 'no such factor' — it will produce the most *plausible-sounding* answer, confidently.

⚠️ This exact failure mode reappears **inside your agent** in notebook 04 — and you will fix it with a guardrail tool.

In [ ]:
hallucination_prompt = (
    "What is the IRDAI motor tariff factor for hatchbacks under 1000cc? "
    "Give the exact factor value and the section reference."
)
resp = client.models.generate_content(model=MODEL, contents=hallucination_prompt)
print(resp.text)
print("\n⚠️  Verify before you trust: there is no such published factor. "
      "Whatever appears above was constructed to be plausible, not true.")

## §5 · Structured output — because agents speak JSON
One config line guarantees parseable JSON. This is how every component of an agentic system exchanges data — prose is only for humans at the last step.

In [ ]:
import json

prompt = """For private car comprehensive insurance, list 5 rating factors.
For each: name, direction (increase/decrease premium), one-line justification. Return JSON."""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
factors = json.loads(resp.text)   # guaranteed to parse
for f in factors:
    print(f)

## §6 · Review exercise — mark the model's homework
Treat the JSON above as a junior analyst's first draft and grade it:

1. Is every **direction** consistent with your priors?
2. Did it name factors your book doesn't collect (e.g. telematics, annual mileage)?
3. What material factors are **missing** (vehicle make? segment?)
4. What would you still need before any of this goes near a tariff filing? *(Hint: magnitudes → a GLM run → notebook 02.)*

**The rule that survives today:** the reasoner narrates; tools know; humans sign.

---
**Log what you ran.** For anything regulatory: save the full prompt–response pair, the model id, and the timestamp — 'the AI wrote it' is not a defence without the receipt.

In [ ]:
# Minimal call log — one CSV row per call. In production this is your observability stack.
import datetime, csv, pathlib

def log_call(prompt, response_text, model=MODEL, path="genai_call_log.csv"):
    new = not pathlib.Path(path).exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["ts_utc", "model", "prompt", "response"])
        w.writerow([datetime.datetime.utcnow().isoformat(), model, prompt, response_text])

log_call(prompt, resp.text)
print("logged — this habit is checklist question 10 in miniature")